# Caption Cleaning

Notebook ini membersihkan kolom `Caption` dari `dataset/labels_skema2.xlsx` dan menghasilkan tiga kolom baru:

- `caption_clean`: caption dibersihkan ringan, hashtag/mention/link dipisahkan atau dihapus dari teks utama.
- `hashtags_clean`: hashtag diekstrak, tanda `#` dihapus, lowercase, dan token camel case/underscore dipisahkan.
- `caption_normalized`: caption clean dengan emoji dan slang dinormalisasi memakai library bila tersedia.

Dependency yang direkomendasikan untuk normalisasi penuh:

```python
%pip install emoji indo-normalizer
```

Hasil akhirnya disimpan ke `caption-preprocessing/caption-clean.csv`.


In [ ]:
# Jalankan sekali jika package belum tersedia di kernel notebook ini.
# Setelah install, restart kernel lalu run all.
# %pip install emoji indo-normalizer


In [ ]:
from pathlib import Path
from zipfile import ZipFile
from xml.etree import ElementTree as ET

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "dataset" / "labels_skema2.xlsx").exists():
    ROOT = ROOT.parent

INPUT_PATH = ROOT / "dataset" / "labels_skema2.xlsx"
OUTPUT_PATH = ROOT / "caption-preprocessing" / "caption-clean.csv"

NS = {
    "main": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
    "rel": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
}


def _xml_text(element):
    if element is None:
        return ""
    return "".join(text_node.text or "" for text_node in element.findall(".//main:t", NS))


def _read_shared_strings(workbook_zip):
    if "xl/sharedStrings.xml" not in workbook_zip.namelist():
        return []
    root = ET.fromstring(workbook_zip.read("xl/sharedStrings.xml"))
    return [_xml_text(item) for item in root.findall("main:si", NS)]


def _column_index(cell_reference):
    letters = "".join(character for character in cell_reference if character.isalpha())
    index = 0
    for letter in letters:
        index = index * 26 + ord(letter.upper()) - ord("A") + 1
    return index - 1


def _cell_value(cell, shared_strings):
    cell_type = cell.attrib.get("t")
    value_node = cell.find("main:v", NS)

    if cell_type == "s":
        if value_node is None or value_node.text is None:
            return ""
        return shared_strings[int(value_node.text)]

    if cell_type == "inlineStr":
        return _xml_text(cell.find("main:is", NS))

    if cell_type == "str":
        return value_node.text if value_node is not None and value_node.text is not None else ""

    return value_node.text if value_node is not None and value_node.text is not None else ""


def _first_sheet_path(workbook_zip):
    workbook_root = ET.fromstring(workbook_zip.read("xl/workbook.xml"))
    first_sheet = workbook_root.find("main:sheets/main:sheet", NS)
    if first_sheet is None:
        raise ValueError("Workbook tidak memiliki sheet.")

    relationship_id = first_sheet.attrib[f"{{{NS['rel']}}}id"]
    rels_root = ET.fromstring(workbook_zip.read("xl/_rels/workbook.xml.rels"))
    for relationship in rels_root:
        if relationship.attrib.get("Id") == relationship_id:
            target = relationship.attrib["Target"].lstrip("/")
            return target if target.startswith("xl/") else f"xl/{target}"

    raise ValueError("Path sheet pertama tidak ditemukan.")


def _unique_headers(headers):
    counts = {}
    unique = []
    for index, header in enumerate(headers):
        name = str(header).strip() or f"Unnamed: {index}"
        counts[name] = counts.get(name, 0) + 1
        unique.append(name if counts[name] == 1 else f"{name}.{counts[name] - 1}")
    return unique


def read_xlsx_first_sheet(path):
    path = Path(path)
    with ZipFile(path) as workbook_zip:
        shared_strings = _read_shared_strings(workbook_zip)
        sheet_path = _first_sheet_path(workbook_zip)
        sheet_root = ET.fromstring(workbook_zip.read(sheet_path))

    rows = []
    max_columns = 0
    for row in sheet_root.findall(".//main:sheetData/main:row", NS):
        values = []
        for cell in row.findall("main:c", NS):
            column_index = _column_index(cell.attrib["r"])
            while len(values) <= column_index:
                values.append("")
            values[column_index] = _cell_value(cell, shared_strings)
        max_columns = max(max_columns, len(values))
        rows.append(values)

    if not rows:
        return pd.DataFrame()

    rows = [row + [""] * (max_columns - len(row)) for row in rows]
    headers = _unique_headers(rows[0])
    data = [row + [""] * (len(headers) - len(row)) for row in rows[1:]]
    df = pd.DataFrame(data, columns=headers)

    empty_unnamed_columns = [
        column
        for column in df.columns
        if str(column).startswith("Unnamed:") and df[column].astype(str).str.strip().eq("").all()
    ]
    return df.drop(columns=empty_unnamed_columns)


In [ ]:
import csv
import re
import unicodedata
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path

try:
    import emoji as emoji_lib
except ImportError:
    emoji_lib = None

try:
    from indo_normalizer import Normalizer
except ImportError:
    Normalizer = None

try:
    from indoNLP.preprocessing import emoji_to_words as indonlp_emoji_to_words
    from indoNLP.preprocessing import replace_slang as indonlp_replace_slang
except ImportError:
    indonlp_emoji_to_words = None
    indonlp_replace_slang = None

try:
    with redirect_stdout(StringIO()):
        INDO_NORMALIZER = Normalizer() if Normalizer is not None else None
except Exception:
    INDO_NORMALIZER = None

HASHTAG_PATTERN = re.compile(r"#([0-9A-Za-z\u00C0-\u00FF_]+)", flags=re.UNICODE)
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
MENTION_PATTERN = re.compile(r"(?<!\w)@[\w_.]+", flags=re.UNICODE)
SPACE_PATTERN = re.compile(r"\s+")
DECORATION_PATTERN = re.compile(
    r"[_=]{2,}|[\u2022\u25CF\u25AA\u25AB\u25A0\u25A1\u25C6\u25C7\u2666\u25E6]+|"
    r"[\u2796]+|[\u2014\u2013\u2212]+|[-]{2,}|[\u00B0]+"
)
VARIATION_AND_MODIFIER_PATTERN = re.compile("[\ufe0f\U0001F3FB-\U0001F3FF]")
DASH_TRANSLATION = str.maketrans({"\u2014": " ", "\u2013": " ", "\u2212": " "})


def load_indo_normalizer_slang_map():
    if INDO_NORMALIZER is None:
        return {}

    slang_path = getattr(INDO_NORMALIZER, "slangs_csv_path", None)
    if not slang_path:
        return getattr(INDO_NORMALIZER, "_SLANG_TO_FORMAL_MAP", {}) or {}

    with Path(slang_path).open("r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        return {
            row["slang"].strip().lower(): row["formal"].strip().lower()
            for row in reader
            if row.get("slang") and row.get("formal")
        }


INDO_NORMALIZER_SLANG_MAP = load_indo_normalizer_slang_map()


def preprocessing_backend_status():
    emoji_backend = "emoji" if emoji_lib is not None else "indoNLP" if indonlp_emoji_to_words is not None else "fallback remove-only"
    slang_backend = (
        "indo-normalizer slang corpus"
        if INDO_NORMALIZER_SLANG_MAP
        else "indoNLP"
        if indonlp_replace_slang is not None
        else "fallback no slang normalization"
    )
    print(f"Emoji backend: {emoji_backend}")
    print(f"Slang backend: {slang_backend}")
    if emoji_lib is None or not INDO_NORMALIZER_SLANG_MAP:
        print("Untuk normalisasi penuh, jalankan: %pip install emoji indo-normalizer")


def normalize_unicode(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except NameError:
        pass
    text = unicodedata.normalize("NFKC", str(value))
    return text.replace("\xa0", " ")


def clean_hashtag_token(tag):
    tag = unicodedata.normalize("NFKC", str(tag)).strip("_")
    tag = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", tag)
    tag = tag.replace("_", " ")
    tag = re.sub(r"[^0-9A-Za-z\u00C0-\u00FF\s]", " ", tag)
    return SPACE_PATTERN.sub(" ", tag).strip().lower()


def extract_hashtags(text):
    text = normalize_unicode(text)
    hashtags = []
    seen = set()
    for match in HASHTAG_PATTERN.findall(text):
        hashtag = clean_hashtag_token(match)
        if hashtag and hashtag not in seen:
            hashtags.append(hashtag)
            seen.add(hashtag)
    return " ".join(hashtags)


def clean_caption_light(text):
    text = normalize_unicode(text).lower()
    text = URL_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = text.replace("&", " dan ")
    text = text.translate(DASH_TRANSLATION)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"(?:\s*\.\s*){2,}", " ", text)
    text = DECORATION_PATTERN.sub(" ", text)
    text = re.sub(r"([!?.,]){2,}", r"\1", text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = SPACE_PATTERN.sub(" ", text)
    return text.strip(" ,.;:-")


def _replace_shortcode_delimiters(text):
    text = re.sub(r":([A-Za-z0-9_+\-]+):", lambda match: " " + match.group(1).replace("_", " ").replace("-", " ") + " ", text)
    text = re.sub(r"!([A-Za-z0-9_+\-]+)!", lambda match: " " + match.group(1).replace("_", " ").replace("-", " ") + " ", text)
    return text


def _strip_unhandled_emoji(text):
    cleaned_characters = []
    for character in text:
        codepoint = ord(character)
        is_emoji_block = (
            0x1F000 <= codepoint <= 0x1FAFF
            or 0x2600 <= codepoint <= 0x27BF
        )
        cleaned_characters.append(" " if is_emoji_block else character)
    return "".join(cleaned_characters)


def replace_emoji_with_words(text):
    text = VARIATION_AND_MODIFIER_PATTERN.sub("", text)

    if emoji_lib is not None:
        try:
            text = emoji_lib.demojize(text, language="id", delimiters=(" ", " "))
        except Exception:
            text = emoji_lib.demojize(text, delimiters=(" ", " "))
        return _replace_shortcode_delimiters(text)

    if indonlp_emoji_to_words is not None:
        return _replace_shortcode_delimiters(indonlp_emoji_to_words(text))

    return _strip_unhandled_emoji(text)


def normalize_repeated_letters(text):
    return re.sub(r"([A-Za-z\u00C0-\u00FF])\1{2,}", r"\1\1", text)


def normalize_slang(text):
    if INDO_NORMALIZER_SLANG_MAP:
        normalized_words = []
        for word in text.split():
            normalized_words.append(INDO_NORMALIZER_SLANG_MAP.get(word, word))
        return " ".join(normalized_words)

    if indonlp_replace_slang is not None:
        return indonlp_replace_slang(text)

    return text


def normalize_caption(text):
    text = normalize_unicode(text).lower()
    text = replace_emoji_with_words(text)
    text = text.replace("&", " dan ")
    text = re.sub(r"\bu\s*/\s*\b", " untuk ", text)
    text = re.sub(r"[^0-9A-Za-z\u00C0-\u00FF\s]", " ", text)
    text = SPACE_PATTERN.sub(" ", text).strip()
    text = normalize_repeated_letters(text)
    text = normalize_slang(text)
    return SPACE_PATTERN.sub(" ", str(text)).strip().lower()


def build_preprocessed_dataframe(df):
    if "Caption" not in df.columns:
        raise KeyError("Kolom 'Caption' tidak ditemukan di dataset.")

    result = df.copy()
    result["caption_clean"] = result["Caption"].apply(clean_caption_light)
    result["hashtags_clean"] = result["Caption"].apply(extract_hashtags)
    result["caption_normalized"] = result["caption_clean"].apply(normalize_caption)
    return result


In [ ]:
df = read_xlsx_first_sheet(INPUT_PATH)
print(f"Dataset shape: {df.shape}")
print(df.columns.tolist())
df.head()


In [ ]:
preprocessing_backend_status()
caption_df = build_preprocessed_dataframe(df)
caption_df[["Caption", "caption_clean", "hashtags_clean", "caption_normalized"]].head(10)


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
try:
    caption_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
    print(f"Saved {len(caption_df)} rows to {OUTPUT_PATH}")
except PermissionError as error:
    print(f"Gagal menyimpan ke {OUTPUT_PATH} karena file sedang terbuka/terkunci.")
    print("Tutup file CSV dari Excel atau aplikasi lain, lalu jalankan ulang cell ini.")
    raise error
